# Stage 2: Ranking with LightGBM LambdaRank

**Pipeline:**
- Stage 1 (Retrieval) -> 300 candidates per user (from pkl file)
- Stage 2 (Ranking) -> Re-rank using LightGBM LambdaRank + Semantic Features -> Top K items

In [1]:
import json
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
import pickle
from tqdm import tqdm
import warnings
import os
import gc
import time
from scipy import sparse
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import lightgbm as lgb

warnings.filterwarnings('ignore')

print("[OK] All libraries imported successfully!")

[OK] All libraries imported successfully!


## 1. Load Data and Retrieval Candidates

In [2]:
# Load review data
review_file = '../data/Luxury_Beauty_5.json'
print(f"Reading reviews from: {review_file}")

reviews = []
with open(review_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Loading reviews"):
        try:
            review = json.loads(line.strip())
            reviews.append({
                'user_id': review.get('reviewerID'),
                'item_id': review.get('asin'),
                'rating': review.get('overall'),
                'timestamp': review.get('unixReviewTime'),
                'review_text': review.get('reviewText', ''),
                'summary': review.get('summary', '')
            })
        except:
            pass

df_reviews = pd.DataFrame(reviews)
print(f"[OK] Loaded {len(df_reviews):,} raw reviews")

# Deduplicate (same review crawled multiple times)
before_dedup = len(df_reviews)
df_reviews = df_reviews.drop_duplicates(subset=['user_id', 'item_id', 'timestamp'], keep='first')
df_reviews = df_reviews.reset_index(drop=True)
print(f"[OK] After dedup: {len(df_reviews):,} reviews (removed {before_dedup - len(df_reviews):,} duplicates)")

Reading reviews from: ../data/Luxury_Beauty_5.json


Loading reviews: 34278it [00:00, 149690.36it/s]

[OK] Loaded 34,278 raw reviews
[OK] After dedup: 29,059 reviews (removed 5,219 duplicates)


In [3]:
# Load metadata
meta_file = '../data/meta_Luxury_Beauty.json'
print(f"Reading metadata from: {meta_file}")

metadata = []
with open(meta_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Loading metadata"):
        try:
            meta = json.loads(line.strip())
            # Extract price if available
            price_str = meta.get('price', '')
            price = None
            if price_str and price_str.startswith('$'):
                try:
                    price = float(price_str.replace('$', '').replace(',', ''))
                except:
                    pass

            # Extract rank
            rank_str = meta.get('rank', '')
            rank = None
            if rank_str:
                try:
                    rank = int(''.join(filter(str.isdigit, rank_str.split(' ')[0])))
                except:
                    pass

            metadata.append({
                'item_id': meta.get('asin'),
                'title': meta.get('title', ''),
                'brand': meta.get('brand', ''),
                'price': price,
                'rank': rank,
                'category': meta.get('category', []),
                'also_buy': meta.get('also_buy', []),
                'also_view': meta.get('also_view', []),
                'description': meta.get('description', []),
                'main_cat': meta.get('main_cat', '')
            })
        except:
            pass

df_meta = pd.DataFrame(metadata)
print(f"[OK] Loaded {len(df_meta):,} product metadata")

Reading metadata from: ../data/meta_Luxury_Beauty.json


Loading metadata: 12299it [00:00, 38975.17it/s]

[OK] Loaded 12,299 product metadata


In [4]:
# Load retrieval candidates from pkl (output of retrieval stage)
# Format: {user_id: [(item_id, score), ...]}  — 300 candidates per user
candidates_file = '../model/hybrid_huim_candidates.pkl'
print(f"Loading retrieval candidates from: {candidates_file}")
with open(candidates_file, 'rb') as f:
    user_candidates_dict = pickle.load(f)
print(f"[OK] Loaded candidates for {len(user_candidates_dict):,} users")

# Show sample
sample_uid = list(user_candidates_dict.keys())[0]
sample_cands = user_candidates_dict[sample_uid]
print(f"  - Sample user: {sample_uid}")
print(f"  - Candidates: {len(sample_cands)} items")
print(f"  - Format: (item_id, score) = {sample_cands[0]}")

Loading retrieval candidates from: ../model/hybrid_huim_candidates.pkl
[OK] Loaded candidates for 2,782 users
  - Sample user: A0488385844WNV2OWO9X
  - Candidates: 300 items
  - Format: (item_id, score) = ('B000CRQHNU', 0.8604337373107691)


In [5]:
model_info_path = '../model/model_info_luxury_beauty.json'
if os.path.exists(model_info_path):
    with open(model_info_path, 'r') as f:
        model_info = json.load(f)
    print(f"[OK] Model info loaded")
else:
    model_info = {'dataset': {}}
    print("[INFO] model_info file not found, will create new one")

# Load semantic embeddings (from semantic_features.py)
semantic_emb_path = '../model/semantic_embeddings.pkl'
if os.path.exists(semantic_emb_path):
    with open(semantic_emb_path, 'rb') as f:
        semantic_data = pickle.load(f)
    user_embeddings = semantic_data['user_embeddings']
    item_embeddings = semantic_data['item_embeddings']
    print(f"[OK] Loaded semantic embeddings: {len(user_embeddings)} users, {len(item_embeddings)} items ({semantic_data['model_name']}, {semantic_data['embedding_dim']}d)")
else:
    user_embeddings = {}
    item_embeddings = {}
    print("[WARNING] semantic_embeddings.pkl not found! Run semantic_features.py first. Continuing without semantic features.")

[OK] Model info loaded
[OK] Loaded semantic embeddings: 2782 users, 1396 items (all-MiniLM-L6-v2, 384d)


## 2. Data Preprocessing

In [6]:
df = df_reviews.copy()
df = df.sort_values(['timestamp', 'item_id']).reset_index(drop=True)
df = df.dropna(subset=['user_id', 'item_id', 'timestamp'])

# Apply same filtering as retrieval
min_user_interactions = 5
min_item_interactions = 3

print(f"Filtering users with >= {min_user_interactions} interactions...")
user_counts = df['user_id'].value_counts()
valid_users = user_counts[user_counts >= min_user_interactions].index
df = df[df['user_id'].isin(valid_users)]

print(f"Filtering items with >= {min_item_interactions} interactions...")
item_counts = df['item_id'].value_counts()
valid_items = item_counts[item_counts >= min_item_interactions].index
df = df[df['item_id'].isin(valid_items)]

df = df.reset_index(drop=True)

print(f"\nAfter filtering:")
print(f"- Total interactions: {len(df):,}")
print(f"- Unique users: {df['user_id'].nunique():,}")
print(f"- Unique items: {df['item_id'].nunique():,}")


Filtering users with >= 5 interactions...
Filtering items with >= 3 interactions...

After filtering:
- Total interactions: 25,206
- Unique users: 2,782
- Unique items: 1,399


## 3. Train/Test Split (Leave-One-Out / Leave-Two-Out)

In [7]:
def create_train_test_split(df):
    """Create train/test split - matching retrieval stage.
    Leave-one-out for users with <= 7 interactions,
    Leave-two-out for users with > 7 interactions.
    """
    print("\nCreating train/test split (leave-one-out / leave-two-out)...")
    df = df.sort_values(['user_id', 'timestamp', 'item_id'])

    train_data = []
    test_data = []

    for user_id, user_df in tqdm(df.groupby('user_id'), desc="Splitting data"):
        n = len(user_df)
        if n <= 7:
            n_test = 1
        else:
            n_test = 2

        train_data.append(user_df.iloc[:-n_test])
        test_items = user_df.iloc[-n_test:]['item_id'].tolist()
        test_data.append({
            'user_id': user_id,
            'ground_truth_items': test_items
        })

    train_df = pd.concat(train_data, ignore_index=True)
    test_df = pd.DataFrame(test_data)

    avg_test_items = test_df['ground_truth_items'].apply(len).mean()
    return train_df, test_df, avg_test_items

train_df, test_df, avg_test_items = create_train_test_split(df)

print(f"\n[OK] Train set: {len(train_df):,} interactions")
print(f"[OK] Test users: {len(test_df):,}")
print(f"[OK] Average test items per user: {avg_test_items:.2f}")

# Check overlap with candidates dict
test_users_in_pkl = sum(1 for uid in test_df['user_id'] if uid in user_candidates_dict)
print(f"[OK] Test users with candidates in pkl: {test_users_in_pkl}/{len(test_df)}")



Creating train/test split (leave-one-out / leave-two-out)...


Splitting data: 100%|██████████| 2782/2782 [00:00<00:00, 9586.27it/s] 


[OK] Train set: 21,076 interactions
[OK] Test users: 2,782
[OK] Average test items per user: 1.48
[OK] Test users with candidates in pkl: 2782/2782


## 4. Feature Engineering

In [8]:
class FeatureBuilder:
    """
    Build features for ranking model

    Features:
    1. User features: interaction count, avg rating, rating std, active days
    2. Item features: popularity, avg rating, price, brand popularity
    3. User-Item features: retrieval scores, co-occurrence
    """

    def __init__(self, train_df, df_meta, user_embeddings=None, item_embeddings=None):
        self.train_df = train_df
        self.df_meta = df_meta
        self.user_embeddings = user_embeddings or {}
        self.item_embeddings = item_embeddings or {}
        self._build_statistics()

    def _build_statistics(self):
        print("\nBuilding statistics...")

        # User statistics
        print("  - User statistics...")
        user_stats = self.train_df.groupby('user_id').agg({
            'item_id': 'count',
            'rating': ['mean', 'std'],
            'timestamp': ['min', 'max']
        })
        user_stats.columns = ['user_interaction_count', 'user_avg_rating', 'user_rating_std',
                              'user_first_interaction', 'user_last_interaction']
        user_stats['user_active_days'] = (user_stats['user_last_interaction'] -
                                           user_stats['user_first_interaction']) / (24*3600)
        user_stats['user_rating_std'] = user_stats['user_rating_std'].fillna(0)
        self.user_stats = user_stats

        # Item statistics
        print("  - Item statistics...")
        item_stats = self.train_df.groupby('item_id').agg({
            'user_id': 'count',
            'rating': ['mean', 'std']
        })
        item_stats.columns = ['item_popularity', 'item_avg_rating', 'item_rating_std']
        item_stats['item_rating_std'] = item_stats['item_rating_std'].fillna(0)

        # Add log_popularity and is_popular_item
        item_stats['log_popularity'] = np.log1p(item_stats['item_popularity'])
        self.median_popularity = item_stats['item_popularity'].median()
        item_stats['is_popular_item'] = (item_stats['item_popularity'] > self.median_popularity).astype(int)

        self.item_stats = item_stats

        # Add metadata features
        print("  - Metadata features...")
        meta_features = self.df_meta[['item_id', 'price', 'rank', 'brand']].copy()

        # Brand popularity
        brand_counts = self.df_meta['brand'].value_counts()
        meta_features['brand_popularity'] = meta_features['brand'].map(brand_counts).fillna(0)

        # Also_buy and also_view counts
        self.df_meta['also_buy_count'] = self.df_meta['also_buy'].apply(len)
        self.df_meta['also_view_count'] = self.df_meta['also_view'].apply(len)
        meta_features = meta_features.merge(
            self.df_meta[['item_id', 'also_buy_count', 'also_view_count']],
            on='item_id', how='left'
        )

        self.meta_features = meta_features.set_index('item_id')

        # User-Item co-purchase matrix
        print("  - Co-purchase statistics...")
        self.user_items = defaultdict(set)
        for _, row in self.train_df.iterrows():
            self.user_items[row['user_id']].add(row['item_id'])

        # Item co-occurrence
        self.item_cooccurrence = defaultdict(Counter)
        for user_id, items in tqdm(self.user_items.items(), desc="Building co-occurrence"):
            items_list = list(items)
            for i, item1 in enumerate(items_list):
                for item2 in items_list[i+1:]:
                    self.item_cooccurrence[item1][item2] += 1
                    self.item_cooccurrence[item2][item1] += 1

        # Global statistics
        self.global_avg_rating = self.train_df['rating'].mean()
        self.max_popularity = item_stats['item_popularity'].max()

        # Price percentile
        print("  - Price statistics...")
        valid_prices = self.meta_features[self.meta_features['price'].notna() & (self.meta_features['price'] > 0)]['price']
        self.price_percentiles = valid_prices.rank(pct=True).to_dict()
        self.median_price = valid_prices.median() if len(valid_prices) > 0 else 0

        # Category mapping for items
        print("  - Category mapping...")
        self.item_categories = {}
        for _, row in self.df_meta.iterrows():
            item_id = row['item_id']
            categories = row.get('category', [])
            if isinstance(categories, list):
                self.item_categories[item_id] = set(categories)
            else:
                self.item_categories[item_id] = set()

        # User category preferences
        print("  - User category preferences...")
        self.user_categories = defaultdict(Counter)
        for _, row in self.train_df.iterrows():
            user_id = row['user_id']
            item_id = row['item_id']
            if item_id in self.item_categories:
                for cat in self.item_categories[item_id]:
                    self.user_categories[user_id][cat] += 1

        # Also-buy sets for overlap calculation
        print("  - Also-buy mapping...")
        self.item_also_buy = {}
        for _, row in self.df_meta.iterrows():
            item_id = row['item_id']
            also_buy = row.get('also_buy', [])
            if isinstance(also_buy, list):
                self.item_also_buy[item_id] = set(also_buy)
            else:
                self.item_also_buy[item_id] = set()

        # User's recent interactions with timestamps
        print("  - User recency mapping...")
        self.user_item_timestamps = defaultdict(dict)
        self.max_timestamp = self.train_df['timestamp'].max()
        for _, row in self.train_df.iterrows():
            self.user_item_timestamps[row['user_id']][row['item_id']] = row['timestamp']

        # Precompute user purchase embeddings for item-to-item semantic features
        if self.item_embeddings:
            print("  - Precomputing user purchase embeddings...")
            self.user_purchase_emb_matrix = {}  # user_id \u2192 (n, dim) normalized
            self.user_last_item_emb = {}  # user_id \u2192 (dim,) normalized

            for user_id, items in self.user_items.items():
                # Collect normalized item embeddings for this user's purchased items
                item_embs = []
                for iid in items:
                    if iid in self.item_embeddings:
                        emb = self.item_embeddings[iid]
                        norm = np.linalg.norm(emb)
                        if norm > 0:
                            item_embs.append(emb / norm)

                if item_embs:
                    self.user_purchase_emb_matrix[user_id] = np.array(item_embs)

                # Last purchased item by timestamp
                user_ts = self.user_item_timestamps.get(user_id, {})
                if user_ts:
                    last_item = max(user_ts, key=user_ts.get)
                    if last_item in self.item_embeddings:
                        emb = self.item_embeddings[last_item]
                        norm = np.linalg.norm(emb)
                        if norm > 0:
                            self.user_last_item_emb[user_id] = emb / norm

            print(f"     Purchase emb matrices: {len(self.user_purchase_emb_matrix)} users")
            print(f"     Last item embs: {len(self.user_last_item_emb)} users")
        else:
            self.user_purchase_emb_matrix = {}
            self.user_last_item_emb = {}

        print("[OK] Statistics built")

    def get_user_features(self, user_id):
        """Get features for a user"""
        if user_id in self.user_stats.index:
            stats = self.user_stats.loc[user_id]
            return {
                'user_interaction_count': stats['user_interaction_count'],
                'user_avg_rating': stats['user_avg_rating'],
                'user_rating_std': stats['user_rating_std'],
                'user_active_days': stats['user_active_days']
            }
        else:
            return {
                'user_interaction_count': 0,
                'user_avg_rating': self.global_avg_rating,
                'user_rating_std': 0,
                'user_active_days': 0
            }

    def get_item_features(self, item_id):
        """Get features for an item"""
        features = {}

        # Item statistics
        if item_id in self.item_stats.index:
            stats = self.item_stats.loc[item_id]
            if isinstance(stats, pd.DataFrame):
                stats = stats.iloc[0]
            features['item_popularity'] = float(stats['item_popularity'])
            features['item_popularity_norm'] = float(stats['item_popularity']) / self.max_popularity
            features['item_avg_rating'] = float(stats['item_avg_rating'])
            features['item_rating_std'] = float(stats['item_rating_std'])
            features['log_popularity'] = float(stats['log_popularity'])
            features['is_popular_item'] = int(stats['is_popular_item'])
        else:
            features['item_popularity'] = 0
            features['item_popularity_norm'] = 0
            features['item_avg_rating'] = self.global_avg_rating
            features['item_rating_std'] = 0
            features['log_popularity'] = 0
            features['is_popular_item'] = 0

        # Metadata features
        if item_id in self.meta_features.index:
            meta = self.meta_features.loc[item_id]
            if isinstance(meta, pd.DataFrame):
                meta = meta.iloc[0]

            price_val = meta['price']
            rank_val = meta['rank']
            brand_pop = meta['brand_popularity']
            also_buy = meta['also_buy_count']
            also_view = meta['also_view_count']

            features['item_price'] = float(price_val) if pd.notna(price_val) else -1
            features['item_rank'] = float(rank_val) if pd.notna(rank_val) else -1
            features['brand_popularity'] = float(brand_pop) if pd.notna(brand_pop) else 0
            features['also_buy_count'] = float(also_buy) if pd.notna(also_buy) else 0
            features['also_view_count'] = float(also_view) if pd.notna(also_view) else 0

            # Price percentile
            if item_id in self.price_percentiles:
                features['price_percentile'] = float(self.price_percentiles[item_id])
            else:
                features['price_percentile'] = 0.5

            # Price category
            if pd.notna(price_val) and price_val > 0:
                features['is_premium_item'] = 1 if price_val > self.median_price * 1.5 else 0
            else:
                features['is_premium_item'] = 0
        else:
            features['item_price'] = -1
            features['item_rank'] = -1
            features['brand_popularity'] = 0
            features['also_buy_count'] = 0
            features['also_view_count'] = 0
            features['price_percentile'] = 0.5
            features['is_premium_item'] = 0

        return features

    def get_user_item_features(self, user_id, item_id):
        """Get user-item interaction features"""
        features = {}

        # Co-occurrence score
        user_items = self.user_items.get(user_id, set())
        cooc_score = 0
        for user_item in user_items:
            if user_item in self.item_cooccurrence:
                cooc_score += self.item_cooccurrence[user_item].get(item_id, 0)
        features['cooccurrence_score'] = cooc_score

        # Number of user's items that co-occur with candidate
        features['num_cooccur_items'] = sum(1 for ui in user_items
                                            if item_id in self.item_cooccurrence.get(ui, {}))

        # Category match score
        user_cats = self.user_categories.get(user_id, Counter())
        item_cats = self.item_categories.get(item_id, set())
        if user_cats and item_cats:
            category_match = sum(user_cats.get(cat, 0) for cat in item_cats)
            total_user_cats = sum(user_cats.values())
            features['category_match_score'] = category_match / max(total_user_cats, 1)
            features['category_overlap_count'] = len(set(user_cats.keys()) & item_cats)
        else:
            features['category_match_score'] = 0
            features['category_overlap_count'] = 0

        # Also-buy overlap
        item_also_buy = self.item_also_buy.get(item_id, set())
        if user_items and item_also_buy:
            overlap = len(user_items & item_also_buy)
            features['also_buy_overlap'] = overlap
            features['also_buy_overlap_ratio'] = overlap / max(len(user_items), 1)
        else:
            features['also_buy_overlap'] = 0
            features['also_buy_overlap_ratio'] = 0

        # Recency weight based on user's recent interactions
        user_timestamps = self.user_item_timestamps.get(user_id, {})
        if user_timestamps:
            recency_weights = []
            for ui in user_items:
                if ui in self.item_cooccurrence and item_id in self.item_cooccurrence[ui]:
                    ts = user_timestamps.get(ui, 0)
                    if ts > 0:
                        days_ago = (self.max_timestamp - ts) / (24 * 3600)
                        recency_weight = 1.0 / (1.0 + days_ago / 365)
                        recency_weights.append(recency_weight)
            features['recency_weighted_cooc'] = np.mean(recency_weights) if recency_weights else 0
        else:
            features['recency_weighted_cooc'] = 0

        return features

    def build_features(self, user_id, item_id, retrieval_rank=None, retrieval_score=None):
        """Build all features for a user-item pair"""
        features = {}

        # User features
        features.update(self.get_user_features(user_id))

        # Item features
        features.update(self.get_item_features(item_id))

        # User-Item features
        features.update(self.get_user_item_features(user_id, item_id))

        # Retrieval features
        if retrieval_rank is not None:
            features['retrieval_rank'] = retrieval_rank
            features['retrieval_rank_inv'] = 1.0 / (retrieval_rank + 1)
        if retrieval_score is not None:
            features['retrieval_score'] = retrieval_score

        # Interaction features (combining user and item features)
        features['rating_interaction'] = features['user_avg_rating'] * features['item_avg_rating']
        features['user_item_rating_diff'] = abs(features['user_avg_rating'] - features['item_avg_rating'])
        features['popularity_ratio'] = features['item_popularity'] / max(features['user_interaction_count'], 1)

        # Semantic features (from Transformer embeddings)
        # 1. User profile \u2192 item cosine similarity
        if user_id in self.user_embeddings and item_id in self.item_embeddings:
            u_emb = self.user_embeddings[user_id]
            i_emb = self.item_embeddings[item_id]
            norm_u = np.linalg.norm(u_emb)
            norm_i = np.linalg.norm(i_emb)
            if norm_u > 0 and norm_i > 0:
                features['semantic_similarity'] = float(np.dot(u_emb, i_emb) / (norm_u * norm_i))
            else:
                features['semantic_similarity'] = 0.0
        else:
            features['semantic_similarity'] = 0.0

        # 2. Item-to-item semantic features (candidate vs user's purchased items)
        if item_id in self.item_embeddings:
            i_emb = self.item_embeddings[item_id]
            norm_i = np.linalg.norm(i_emb)

            if norm_i > 0 and user_id in self.user_purchase_emb_matrix:
                i_emb_norm = i_emb / norm_i
                purchase_matrix = self.user_purchase_emb_matrix[user_id]
                sims = purchase_matrix @ i_emb_norm  # (n,) cosine similarities

                features['semantic_max_sim'] = float(np.max(sims))
                features['semantic_avg_sim'] = float(np.mean(sims))
            else:
                features['semantic_max_sim'] = 0.0
                features['semantic_avg_sim'] = 0.0

            # Similarity to last purchased item
            if norm_i > 0 and user_id in self.user_last_item_emb:
                i_emb_norm = i_emb / norm_i
                features['semantic_last_sim'] = float(np.dot(self.user_last_item_emb[user_id], i_emb_norm))
            else:
                features['semantic_last_sim'] = 0.0
        else:
            features['semantic_max_sim'] = 0.0
            features['semantic_avg_sim'] = 0.0
            features['semantic_last_sim'] = 0.0

        return features

In [9]:
# Create feature builder
feature_builder = FeatureBuilder(train_df, df_meta, user_embeddings, item_embeddings)
print("\n[OK] FeatureBuilder created")
if user_embeddings:
    print(f"     Semantic features: ENABLED ({len(user_embeddings)} user embs, {len(item_embeddings)} item embs)")
    print(f"     Features: semantic_similarity, semantic_max_sim, semantic_avg_sim, semantic_last_sim")
else:
    print("     Semantic features: DISABLED (no embeddings loaded)")


Building statistics...
  - User statistics...
  - Item statistics...
  - Metadata features...
  - Co-purchase statistics...


Building co-occurrence: 100%|██████████| 2782/2782 [00:00<00:00, 33570.94it/s]


  - Price statistics...
  - Category mapping...
  - User category preferences...
  - Also-buy mapping...
  - User recency mapping...
  - Precomputing user purchase embeddings...
     Purchase emb matrices: 2782 users
     Last item embs: 2780 users
[OK] Statistics built

[OK] FeatureBuilder created
     Semantic features: ENABLED (2782 user embs, 1396 item embs)
     Features: semantic_similarity, semantic_max_sim, semantic_avg_sim, semantic_last_sim


## 5. Generate Candidates and Build Training Data

In [10]:
N_CANDIDATES = 300  # Number of candidates from retrieval

def generate_ranking_data(test_df, train_df, user_candidates_dict, feature_builder,
                          n_candidates=300, is_training=True):
    """
    Generate training/test data for ranking model

    For each user:
    1. Get top N candidates from retrieval pkl
    2. Build features for each candidate
    3. Label: 1 if item is in ground_truth_items, 0 otherwise
    """

    # Build user history
    user_train_items = defaultdict(set)
    for _, row in train_df.iterrows():
        user_train_items[row['user_id']].add(row['item_id'])

    all_features = []
    all_labels = []
    all_groups = []  # For LambdaRank
    all_user_ids = []
    all_item_ids = []
    skipped_users = 0

    for _, row in tqdm(test_df.iterrows(), total=len(test_df),
                       desc="Building ranking data"):
        user_id = row['user_id']
        ground_truth = set(row['ground_truth_items']) if is_training else set()

        # Get candidates from pkl dict
        if user_id not in user_candidates_dict:
            skipped_users += 1
            continue

        candidate_tuples = user_candidates_dict[user_id][:n_candidates]
        candidates = [item_id for item_id, score in candidate_tuples]
        candidate_scores = {item_id: score for item_id, score in candidate_tuples}

        # If ground truth items not in candidates (for training), add them
        if is_training:
            for gt_item in ground_truth:
                if gt_item not in candidates:
                    candidates = candidates[:n_candidates-1] + [gt_item]
                    candidate_scores[gt_item] = 0.0

        group_size = 0
        for rank, item_id in enumerate(candidates):
            features = feature_builder.build_features(
                user_id=user_id,
                item_id=item_id,
                retrieval_rank=rank,
                retrieval_score=candidate_scores.get(item_id, 0.0)
            )

            all_features.append(features)
            all_user_ids.append(user_id)
            all_item_ids.append(item_id)

            if is_training:
                label = 1 if item_id in ground_truth else 0
                all_labels.append(label)

            group_size += 1

        all_groups.append(group_size)

    if skipped_users > 0:
        print(f"  [WARNING] Skipped {skipped_users} users not found in candidates pkl")

    # Convert to DataFrame
    features_df = pd.DataFrame(all_features)
    features_df['user_id'] = all_user_ids
    features_df['item_id'] = all_item_ids

    if is_training:
        features_df['label'] = all_labels

    return features_df, all_groups

In [11]:
# Generate training data
print("[1/2] Generating training data...")
train_ranking_df, train_groups = generate_ranking_data(
    test_df,  # Use test_df because we need the actual purchased items
    train_df,
    user_candidates_dict,
    feature_builder,
    n_candidates=N_CANDIDATES,
    is_training=True
)

print(f"\n[OK] Training data generated:")
print(f"  - Total samples: {len(train_ranking_df):,}")
print(f"  - Positive samples: {train_ranking_df['label'].sum():,}")
print(f"  - Negative samples: {(train_ranking_df['label'] == 0).sum():,}")
print(f"  - Number of groups: {len(train_groups):,}")
print(f"  - Features: {len([c for c in train_ranking_df.columns if c not in ['user_id', 'item_id', 'label']])}")

[1/2] Generating training data...


Building ranking data: 100%|██████████| 2782/2782 [14:50<00:00,  3.12it/s]



[OK] Training data generated:
  - Total samples: 834,600
  - Positive samples: 3,856
  - Negative samples: 830,744
  - Number of groups: 2,782
  - Features: 34


## 6. Feature Statistics

In [12]:
feature_cols = [c for c in train_ranking_df.columns if c not in ['user_id', 'item_id', 'label']]
print(f"Feature columns ({len(feature_cols)}):")
for col in feature_cols:
    print(f"  - {col}")

print("\nFeature value ranges:")
print(train_ranking_df[feature_cols].describe().T[['mean', 'std', 'min', 'max']])

Feature columns (34):
  - user_interaction_count
  - user_avg_rating
  - user_rating_std
  - user_active_days
  - item_popularity
  - item_popularity_norm
  - item_avg_rating
  - item_rating_std
  - log_popularity
  - is_popular_item
  - item_price
  - item_rank
  - brand_popularity
  - also_buy_count
  - also_view_count
  - price_percentile
  - is_premium_item
  - cooccurrence_score
  - num_cooccur_items
  - category_match_score
  - category_overlap_count
  - also_buy_overlap
  - also_buy_overlap_ratio
  - recency_weighted_cooc
  - retrieval_rank
  - retrieval_rank_inv
  - retrieval_score
  - rating_interaction
  - user_item_rating_diff
  - popularity_ratio
  - semantic_similarity
  - semantic_max_sim
  - semantic_avg_sim
  - semantic_last_sim

Feature value ranges:
                                 mean            std       min           max
user_interaction_count       7.575845       6.850400  1.000000  9.000000e+01
user_avg_rating              4.258282       0.776178  1.000000  5.00

## 7. Train LightGBM LambdaRank

In [13]:
# Prepare data for LightGBM
X = train_ranking_df[feature_cols].values
y = train_ranking_df['label'].values
groups = train_groups

print(f"Data shape: X={X.shape}, y={y.shape}")
print(f"Number of groups: {len(groups)}")
print(f"Avg group size: {np.mean(groups):.1f}")

# Split into train/validation (by groups)
n_train_groups = int(len(groups) * 0.8)
train_end_idx = sum(groups[:n_train_groups])

X_train, X_val = X[:train_end_idx], X[train_end_idx:]
y_train, y_val = y[:train_end_idx], y[train_end_idx:]
groups_train, groups_val = groups[:n_train_groups], groups[n_train_groups:]

print(f"\nTrain: {len(X_train):,} samples, {len(groups_train):,} groups")
print(f"Val: {len(X_val):,} samples, {len(groups_val):,} groups")

# Create LightGBM datasets
train_data = lgb.Dataset(
    X_train, label=y_train,
    group=groups_train,
    feature_name=feature_cols
)
val_data = lgb.Dataset(
    X_val, label=y_val,
    group=groups_val,
    feature_name=feature_cols,
    reference=train_data
)

# Feature selection - remove zero/low importance features
drop_features = ['brand_popularity', 'is_popular_item', 'log_popularity',
                 'category_match_score', 'category_overlap_count', 'is_premium_item']
feature_cols = [f for f in feature_cols if f not in drop_features]
print(f"\n[Feature Selection] Removed low-importance features: {drop_features}")
print(f"Remaining features ({len(feature_cols)}): {feature_cols}")

# Re-prepare data with selected features
X = train_ranking_df[feature_cols].values
y = train_ranking_df['label'].values

# Re-split
X_train, X_val = X[:train_end_idx], X[train_end_idx:]
y_train, y_val = y[:train_end_idx], y[train_end_idx:]

# Re-create datasets
train_data = lgb.Dataset(X_train, label=y_train, group=groups_train, feature_name=feature_cols)
val_data = lgb.Dataset(X_val, label=y_val, group=groups_val, feature_name=feature_cols, reference=train_data)

Data shape: X=(834600, 34), y=(834600,)
Number of groups: 2782
Avg group size: 300.0

Train: 667,500 samples, 2,225 groups
Val: 167,100 samples, 557 groups

[Feature Selection] Removed low-importance features: ['brand_popularity', 'is_popular_item', 'log_popularity', 'category_match_score', 'category_overlap_count', 'is_premium_item']
Remaining features (28): ['user_interaction_count', 'user_avg_rating', 'user_rating_std', 'user_active_days', 'item_popularity', 'item_popularity_norm', 'item_avg_rating', 'item_rating_std', 'item_price', 'item_rank', 'also_buy_count', 'also_view_count', 'price_percentile', 'cooccurrence_score', 'num_cooccur_items', 'also_buy_overlap', 'also_buy_overlap_ratio', 'recency_weighted_cooc', 'retrieval_rank', 'retrieval_rank_inv', 'retrieval_score', 'rating_interaction', 'user_item_rating_diff', 'popularity_ratio', 'semantic_similarity', 'semantic_max_sim', 'semantic_avg_sim', 'semantic_last_sim']


In [14]:
# LambdaRank parameters
params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [10, 20, 50],
    'boosting_type': 'gbdt',
    'num_leaves': 95,
    'learning_rate': 0.02,
    'feature_fraction': 0.85,
    'bagging_fraction': 0.8,
    'bagging_freq': 2,
    'max_depth': 10,
    'min_child_samples': 10,
    'reg_alpha': 0.05,
    'reg_lambda': 0.05,
    'min_gain_to_split': 0.01,
    'path_smooth': 1.0,
    'verbose': -1,
    'seed': 42,
    'deterministic': True,
    'force_row_wise': True,
    'num_threads': 1
}

print("Training LightGBM LambdaRank...")
print(f"Parameters: {params}")

# Train model
start_time = time.time()
ranking_model = lgb.train(
    params,
    train_data,
    num_boost_round=1200,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),
        lgb.log_evaluation(period=50)
    ]
)
train_time = time.time() - start_time

print(f"\n[OK] Training completed in {train_time:.1f}s")
print(f"  Best iteration: {ranking_model.best_iteration}")

Training LightGBM LambdaRank...
Parameters: {'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [10, 20, 50], 'boosting_type': 'gbdt', 'num_leaves': 95, 'learning_rate': 0.02, 'feature_fraction': 0.85, 'bagging_fraction': 0.8, 'bagging_freq': 2, 'max_depth': 10, 'min_child_samples': 10, 'reg_alpha': 0.05, 'reg_lambda': 0.05, 'min_gain_to_split': 0.01, 'path_smooth': 1.0, 'verbose': -1, 'seed': 42, 'deterministic': True, 'force_row_wise': True, 'num_threads': 1}
Training until validation scores don't improve for 100 rounds
[50]	train's ndcg@10: 0.794934	train's ndcg@20: 0.801204	train's ndcg@50: 0.80894	val's ndcg@10: 0.650214	val's ndcg@20: 0.669693	val's ndcg@50: 0.687068
[100]	train's ndcg@10: 0.818975	train's ndcg@20: 0.824819	train's ndcg@50: 0.833276	val's ndcg@10: 0.654612	val's ndcg@20: 0.676831	val's ndcg@50: 0.694495
[150]	train's ndcg@10: 0.841533	train's ndcg@20: 0.848443	train's ndcg@50: 0.856945	val's ndcg@10: 0.659174	val's ndcg@20: 0.681131	val's ndcg@50: 0.698

## 8. Feature Importance

In [15]:
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': ranking_model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print("Top features by gain:")
print(importance_df.to_string(index=False))

Top features by gain:
               feature    importance
       retrieval_score 301728.437752
     semantic_last_sim 207870.648095
      semantic_max_sim  43052.020006
        retrieval_rank  39231.741801
    cooccurrence_score  32191.557593
   semantic_similarity  30575.100773
    retrieval_rank_inv  25147.133514
 recency_weighted_cooc  21957.107798
             item_rank  21896.016120
      popularity_ratio  18586.822321
also_buy_overlap_ratio  17917.106126
      semantic_avg_sim  17576.063613
       item_popularity  17206.886756
        also_buy_count  14989.494941
      user_active_days  14224.887139
 user_item_rating_diff  12463.423484
       item_rating_std  12292.725081
            item_price  11635.396191
       item_avg_rating  11217.760438
    rating_interaction  11009.142029
       also_view_count   9949.675901
     num_cooccur_items   7036.956379
      price_percentile   6867.592445
       user_rating_std   6825.075461
user_interaction_count   5962.231805
       user_avg_

## 9. Evaluation

In [16]:
def evaluate_ranking(test_df, train_df, user_candidates_dict, ranking_model,
                     feature_builder, feature_cols, n_candidates=300, k_list=[10, 20, 50]):
    """
    Evaluate the two-stage system:
    1. Retrieval: Get N candidates from pkl
    2. Ranking: Re-rank candidates using LightGBM

    Metrics: Recall@K, NDCG@K, MRR@K
    Supports multiple ground truth items per user (Leave-1/Leave-2).
    """

    # Metrics storage
    retrieval_recalls = {k: [] for k in k_list}
    ranking_recalls = {k: [] for k in k_list}
    retrieval_ndcgs = {k: [] for k in k_list}
    ranking_ndcgs = {k: [] for k in k_list}
    retrieval_mrrs = {k: [] for k in k_list}
    ranking_mrrs = {k: [] for k in k_list}

    skipped = 0
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Evaluating"):
        user_id = row['user_id']
        ground_truth = set(row['ground_truth_items'])
        n_gt = len(ground_truth)

        # Get candidates from pkl
        if user_id not in user_candidates_dict:
            skipped += 1
            continue

        candidate_tuples = user_candidates_dict[user_id][:n_candidates]
        candidates = [item_id for item_id, score in candidate_tuples]
        candidate_scores = {item_id: score for item_id, score in candidate_tuples}

        # Stage 2: Ranking - Build features for all candidates
        candidate_features = []
        for rank, item_id in enumerate(candidates):
            features = feature_builder.build_features(
                user_id=user_id,
                item_id=item_id,
                retrieval_rank=rank,
                retrieval_score=candidate_scores.get(item_id, 0.0)
            )
            candidate_features.append(features)

        # Predict ranking scores
        features_matrix = pd.DataFrame(candidate_features)[feature_cols].values
        ranking_scores = ranking_model.predict(features_matrix)

        # Re-rank by scores
        ranked_indices = np.argsort(-ranking_scores)
        reranked_candidates = [candidates[i] for i in ranked_indices]

        # Calculate metrics for both retrieval and ranking
        for k in k_list:
            # --- Retrieval metrics ---
            retrieval_top_k = candidates[:k]
            retrieval_hits = len(ground_truth & set(retrieval_top_k))
            retrieval_recalls[k].append(retrieval_hits / n_gt)

            # MRR: 1 / rank of first hit
            mrr = 0.0
            for i, item in enumerate(retrieval_top_k, 1):
                if item in ground_truth:
                    mrr = 1.0 / i
                    break
            retrieval_mrrs[k].append(mrr)

            # NDCG
            dcg = 0.0
            for i, item in enumerate(retrieval_top_k, 1):
                if item in ground_truth:
                    dcg += 1.0 / np.log2(i + 1)
            idcg = sum(1.0 / np.log2(i + 2) for i in range(min(n_gt, k)))
            retrieval_ndcgs[k].append(dcg / idcg if idcg > 0 else 0.0)

            # --- Ranking metrics ---
            ranking_top_k = reranked_candidates[:k]
            ranking_hits = len(ground_truth & set(ranking_top_k))
            ranking_recalls[k].append(ranking_hits / n_gt)

            mrr = 0.0
            for i, item in enumerate(ranking_top_k, 1):
                if item in ground_truth:
                    mrr = 1.0 / i
                    break
            ranking_mrrs[k].append(mrr)

            dcg = 0.0
            for i, item in enumerate(ranking_top_k, 1):
                if item in ground_truth:
                    dcg += 1.0 / np.log2(i + 1)
            ranking_ndcgs[k].append(dcg / idcg if idcg > 0 else 0.0)

    if skipped > 0:
        print(f"  [WARNING] Skipped {skipped} users not in candidates pkl")

    # Build results
    results = []
    for k in k_list:
        results.append({
            'K': k,
            'Retrieval_Recall': np.mean(retrieval_recalls[k]),
            'Ranking_Recall': np.mean(ranking_recalls[k]),
            'Recall_Improvement': (np.mean(ranking_recalls[k]) - np.mean(retrieval_recalls[k])) / np.mean(retrieval_recalls[k]) * 100 if np.mean(retrieval_recalls[k]) > 0 else 0,
            'Retrieval_NDCG': np.mean(retrieval_ndcgs[k]),
            'Ranking_NDCG': np.mean(ranking_ndcgs[k]),
            'NDCG_Improvement': (np.mean(ranking_ndcgs[k]) - np.mean(retrieval_ndcgs[k])) / np.mean(retrieval_ndcgs[k]) * 100 if np.mean(retrieval_ndcgs[k]) > 0 else 0,
            'Retrieval_MRR': np.mean(retrieval_mrrs[k]),
            'Ranking_MRR': np.mean(ranking_mrrs[k]),
            'MRR_Improvement': (np.mean(ranking_mrrs[k]) - np.mean(retrieval_mrrs[k])) / np.mean(retrieval_mrrs[k]) * 100 if np.mean(retrieval_mrrs[k]) > 0 else 0
        })

    return pd.DataFrame(results)

In [17]:
# Evaluate
k_values = [5, 10, 20, 50]
eval_results = evaluate_ranking(
    test_df, train_df,
    user_candidates_dict, ranking_model,
    feature_builder, feature_cols,
    n_candidates=N_CANDIDATES,
    k_list=k_values
)

print("\n" + "="*60)
print("COMPARISON: RETRIEVAL vs RANKING")
print("="*60)
print("\n", eval_results.to_string(index=False))

# Detailed analysis
print("\n" + "="*60)
print("DETAILED ANALYSIS")
print("="*60)

for _, row in eval_results.iterrows():
    k = int(row['K'])
    print(f"\n--- Top-{k} ---")
    print(f"  Recall:  {row['Retrieval_Recall']:.4f} -> {row['Ranking_Recall']:.4f} ({row['Recall_Improvement']:+.1f}%)")
    print(f"  NDCG:    {row['Retrieval_NDCG']:.4f} -> {row['Ranking_NDCG']:.4f} ({row['NDCG_Improvement']:+.1f}%)")
    print(f"  MRR:     {row['Retrieval_MRR']:.4f} -> {row['Ranking_MRR']:.4f} ({row['MRR_Improvement']:+.1f}%)")

Evaluating: 100%|██████████| 2782/2782 [17:44<00:00,  2.61it/s]


COMPARISON: RETRIEVAL vs RANKING

  K  Retrieval_Recall  Ranking_Recall  Recall_Improvement  Retrieval_NDCG  Ranking_NDCG  NDCG_Improvement  Retrieval_MRR  Ranking_MRR  MRR_Improvement
 5          0.086089        0.534687          521.085595        0.053492      0.523558        878.751052       0.045219     0.567392      1154.756227
10          0.259885        0.563264          116.735823        0.118622      0.533853        350.043937       0.071076     0.571557       704.152635
20          0.320093        0.586988           83.380124        0.135103      0.540474        300.046075       0.076042     0.573266       653.881043
50          0.413192        0.614845           48.803828        0.155825      0.546719        250.853999       0.079974     0.574087       617.839305

DETAILED ANALYSIS

--- Top-5 ---
  Recall:  0.0861 -> 0.5347 (+521.1%)
  NDCG:    0.0535 -> 0.5236 (+878.8%)
  MRR:     0.0452 -> 0.5674 (+1154.8%)

--- Top-10 ---
  Recall:  0.2599 -> 0.5633 (+116.7%)
  NDCG:    

## 10. Save Model

In [18]:
# Save LightGBM model
ranking_model.save_model('../model/ranking_lightgbm_luxury_beauty.txt')
print("[OK] LightGBM model saved to '../model/ranking_lightgbm_luxury_beauty.txt'")

# Save as pickle for easier loading
with open('../model/ranking_model_luxury_beauty.pkl', 'wb') as f:
    pickle.dump({
        'model': ranking_model,
        'feature_cols': feature_cols,
        'params': params
    }, f)
print("[OK] Ranking model (pkl) saved to '../model/ranking_model_luxury_beauty.pkl'")

# Save feature builder
with open('../model/feature_builder_luxury_beauty.pkl', 'wb') as f:
    pickle.dump(feature_builder, f)
print("[OK] Feature builder saved to '../model/feature_builder_luxury_beauty.pkl'")

# Save evaluation results
eval_results.to_csv('../model/ranking_evaluation_HUIM.csv', index=False)
print("[OK] Evaluation results saved to '../model/ranking_evaluation_HUIM.csv'")

# Update model info
model_info['ranking'] = {
    'model': 'LightGBM LambdaRank',
    'n_candidates': N_CANDIDATES,
    'candidates_source': 'hybrid_huim_candidates.pkl',
    'best_iteration': ranking_model.best_iteration,
    'train_time': train_time,
    'feature_cols': feature_cols,
    'metrics': {
        **{f'Recall@{int(row["K"])}': float(row['Ranking_Recall']) for _, row in eval_results.iterrows()},
        **{f'NDCG@{int(row["K"])}': float(row['Ranking_NDCG']) for _, row in eval_results.iterrows()},
        **{f'MRR@{int(row["K"])}': float(row['Ranking_MRR']) for _, row in eval_results.iterrows()}
    }
}

with open('../model/model_info_luxury_beauty.json', 'w') as f:
    json.dump(model_info, f, indent=2)
print("[OK] Model info updated")

print("\n" + "="*60)
print("[DONE] RANKING STAGE COMPLETE!")
print("="*60)
print(f"\nSummary:")
print(f"  - Retrieval: Hybrid Ensemble (CB + CF + Pop) from pkl")
print(f"  - Ranking model: LightGBM LambdaRank")
print(f"  - Candidates per user: {N_CANDIDATES}")
print(f"  - Features: {len(feature_cols)}")
print(f"\nKey improvements over retrieval:")
for _, row in eval_results.iterrows():
    if row['K'] in [10, 20]:
        print(f"  - Recall@{int(row['K'])}: {row['Recall_Improvement']:+.1f}%")
        print(f"  - NDCG@{int(row['K'])}: {row['NDCG_Improvement']:+.1f}%")

[OK] LightGBM model saved to '../model/ranking_lightgbm_luxury_beauty.txt'
[OK] Ranking model (pkl) saved to '../model/ranking_model_luxury_beauty.pkl'
[OK] Feature builder saved to '../model/feature_builder_luxury_beauty.pkl'
[OK] Evaluation results saved to '../model/ranking_evaluation_HUIM.csv'
[OK] Model info updated

[DONE] RANKING STAGE COMPLETE!

Summary:
  - Retrieval: Hybrid Ensemble (CB + CF + Pop) from pkl
  - Ranking model: LightGBM LambdaRank
  - Candidates per user: 300
  - Features: 28

Key improvements over retrieval:
  - Recall@10: +116.7%
  - NDCG@10: +350.0%
  - Recall@20: +83.4%
  - NDCG@20: +300.0%


## 11. Save Ranking Output

In [19]:
def get_recommendations(user_id, user_candidates_dict, ranking_model, feature_builder,
                        feature_cols, n_candidates=300, top_k=20, verbose=False):
    """
    Two-stage recommendation pipeline:
    1. Retrieval: Get N candidates from pkl
    2. Ranking: Re-rank to get top K
    """
    if user_id not in user_candidates_dict:
        if verbose:
            print(f"[WARNING] User {user_id} not found in candidates dict")
        return []

    # Stage 1: Get candidates from pkl
    candidate_tuples = user_candidates_dict[user_id][:n_candidates]
    candidates = [item_id for item_id, score in candidate_tuples]
    candidate_scores = {item_id: score for item_id, score in candidate_tuples}
    if verbose:
        print(f"Stage 1 (Retrieval): Got {len(candidates)} candidates from pkl")

    # Stage 2: Ranking
    candidate_features = []
    for rank, item_id in enumerate(candidates):
        features = feature_builder.build_features(
            user_id=user_id,
            item_id=item_id,
            retrieval_rank=rank,
            retrieval_score=candidate_scores.get(item_id, 0.0)
        )
        candidate_features.append(features)

    features_matrix = pd.DataFrame(candidate_features)[feature_cols].values
    ranking_scores = ranking_model.predict(features_matrix)

    # Re-rank by scores
    ranked_indices = np.argsort(-ranking_scores)[:top_k]
    top_items = [(candidates[i], ranking_scores[i]) for i in ranked_indices]

    if verbose:
        print(f"Stage 2 (Ranking): Re-ranked to top {top_k}")

    return top_items

In [20]:
# Generate top-20 predictions for all test users
user_top20 = {}
test_ground_truth = {}
user_train_items_list = {}

# Build user train items (as list, sorted by time)
for user_id, group in train_df.sort_values('timestamp').groupby('user_id'):
    user_train_items_list[user_id] = group['item_id'].tolist()

# Generate predictions
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Generating top-20"):
    user_id = row['user_id']
    test_ground_truth[user_id] = row['ground_truth_items']

    if user_id not in user_candidates_dict:
        continue

    recs = get_recommendations(
        user_id, user_candidates_dict, ranking_model,
        feature_builder, feature_cols, n_candidates=N_CANDIDATES, top_k=20
    )
    user_top20[user_id] = recs

ranking_output = {
    'user_top20': user_top20,
    'test_ground_truth': test_ground_truth,
    'user_train_items': user_train_items_list,
}
with open('../model/recs_huim_sem.pkl', 'wb') as f:
    pickle.dump(ranking_output, f)
print(f"[OK] Ranking output saved: {len(user_top20)} users with top-20 predictions")
print(f"     File: ../model/recs_huim_sem.pkl")

Generating top-20: 100%|██████████| 2782/2782 [18:27<00:00,  2.51it/s]

[OK] Ranking output saved: 2782 users with top-20 predictions
     File: ../model/recs_huim_sem.pkl


## 12. Inference Example

In [21]:
# Example: Get recommendations for a random user
sample_user = test_df['user_id'].iloc[0]
sample_gt = test_df[test_df['user_id'] == sample_user]['ground_truth_items'].iloc[0]
print(f"User: {sample_user}")
print(f"Ground truth items: {sample_gt}")

recommendations = get_recommendations(
    sample_user,
    user_candidates_dict,
    ranking_model,
    feature_builder,
    feature_cols,
    n_candidates=300,
    top_k=20,
    verbose=True
)

print(f"\nTop 20 recommendations:")
for i, (item_id, score) in enumerate(recommendations, 1):
    # Get item title from metadata
    item_meta = df_meta[df_meta['item_id'] == item_id]
    title = item_meta['title'].iloc[0][:50] if len(item_meta) > 0 else 'Unknown'
    print(f"  {i:2d}. {item_id} (score: {score:.4f}) - {title}...")

User: A0488385844WNV2OWO9X
Ground truth items: ['B00VV8S1GK', 'B00F6XZNOO']
Stage 1 (Retrieval): Got 300 candidates from pkl
Stage 2 (Ranking): Re-ranked to top 20

Top 20 recommendations:
   1. B00VV8S1GK (score: 0.8789) - jane iredale LipDrink Lip Balm...
   2. B00F9JGPTW (score: -0.0431) - jane iredale PureMoist Lipstick...
   3. B0001EKYEM (score: -0.4134) - jane iredale PureLash Lengthening Mascara...
   4. B001P3QHHG (score: -0.6558) - jane iredale Liquid Eyeliner...
   5. B00354QIEU (score: -0.8108) - jane iredale PureMatte Finish Powder Refill...
   6. B001FNPJ5C (score: -0.8117) - Jane Iredale Dream Tint Moisturizer...
   7. B000CQYJ4U (score: -0.8306) - jane iredale PureBrow Brow Gel, 0.17 oz....
   8. B001J3ISOM (score: -0.8518) - jane iredale Active Light Under-eye Concealer, 0.0...
   9. B0002H842C (score: -0.8999) - jane iredale PurePressed Blush, 0.10 oz....
  10. B001FNOA5C (score: -0.9385) - jane iredale Quad Bronzer...
  11. B0001EKVCW (score: -0.9611) - jane iredale 